In [1]:
import findspark
findspark.init()

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

from pyspark.sql import functions as F


**Question 1: Install Spark and PySpark**
- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

In [2]:
#create spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

#spark version
spark.version


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/05 12:36:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.4'

**Question 2: Yellow October 2024**

Read the October 2024 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? 

Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- 100MB

In [3]:
schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True),
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True),
    types.StructField('tpep_dropoff_datetime', types.TimestampType(), True),
    types.StructField('passenger_count', types.IntegerType(), True),
    types.StructField('trip_distance', types.IntegerType(), True),
    types.StructField('RatecodeID', types.IntegerType(), True),
    types.StructField('store_and_fwd_flag', types.StringType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('payment_type', types.StringType(), True),
    types.StructField('fare_amount', types.IntegerType(), True),
    types.StructField('extra', types.StringType(), True),
    types.StructField('mta_tax', types.StringType(), True),
    types.StructField('tip_amount', types.IntegerType(), True),
    types.StructField('tolls_amount', types.IntegerType(), True),
    types.StructField('improvement_surcharge', types.IntegerType(), True),
    types.StructField('total_amount', types.IntegerType(), True),
    types.StructField('congestion_surcharge', types.IntegerType(), True),
    types.StructField('Airport_fee', types.IntegerType(), True)
])

df = spark.read.parquet('yellow_tripdata_2024-10.parquet', schema=schema) \
       .withColumn("tpep_pickup_datetime", F.col("tpep_pickup_datetime").cast("timestamp")) \
       .withColumn("tpep_dropoff_datetime", F.col("tpep_dropoff_datetime").cast("timestamp"))

df= df \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [4]:
output_path = "output_homework"

df \
    .repartition(4) \
    .write.parquet(output_path, mode='overwrite')

In [5]:
!ls -lh output_homework

total 205312
-rw-r--r--@ 1 lorper  staff     0B Mar  5 12:36 _SUCCESS
-rw-r--r--@ 1 lorper  staff    24M Mar  5 12:36 part-00000-2cfb96f5-90a1-4a89-8350-468182fabd42-c000.snappy.parquet
-rw-r--r--@ 1 lorper  staff    24M Mar  5 12:36 part-00001-2cfb96f5-90a1-4a89-8350-468182fabd42-c000.snappy.parquet
-rw-r--r--@ 1 lorper  staff    24M Mar  5 12:36 part-00002-2cfb96f5-90a1-4a89-8350-468182fabd42-c000.snappy.parquet
-rw-r--r--@ 1 lorper  staff    24M Mar  5 12:36 part-00003-2cfb96f5-90a1-4a89-8350-468182fabd42-c000.snappy.parquet


**Q3: Count records**

How many taxi trips were there on the 15th of October?

Consider only trips that started on the 15th of October.

- 85,567
- 105,567
- 125,567
- 145,567


In [6]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .filter("pickup_date = '2024-10-15'") \
    .count()



128893

In [7]:
df.registerTempTable('trip_data')

spark.sql("""
SELECT
    count(1)
FROM 
    trip_data
WHERE
    CAST(pickup_datetime AS DATE) = '2024-10-15'

;
""").show()

/opt/homebrew/Cellar/apache-spark/3.5.4/libexec/python/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


+--------+
|count(1)|
+--------+
|  128893|
+--------+



**Q4: Longest Trip**

What is the length of the longest trip in the dataset in hours?

- 122
- 142
- 162
- 182


In [8]:
df \
    .withColumn('trip_duration(h)', (F.col("dropoff_datetime").cast("long") - F.col('pickup_datetime').cast("long")) / 3600) \
    .sort(F.desc('trip_duration(h)') ) \
    .select(['dropoff_datetime', 'pickup_datetime', 'trip_duration(h)']) \
    .limit(10) \
    .show()

+-------------------+-------------------+------------------+
|   dropoff_datetime|    pickup_datetime|  trip_duration(h)|
+-------------------+-------------------+------------------+
|2024-10-23 07:40:53|2024-10-16 13:03:49|162.61777777777777|
|2024-10-09 18:06:55|2024-10-03 18:47:25|           143.325|
|2024-10-28 09:46:33|2024-10-22 16:00:55|138.76055555555556|
|2024-10-23 04:43:37|2024-10-18 09:53:32|114.83472222222223|
|2024-10-24 18:30:18|2024-10-21 00:36:24| 89.89833333333333|
|2024-10-24 06:57:38|2024-10-20 13:30:52| 89.44611111111111|
|2024-10-25 14:22:49|2024-10-22 16:04:52| 70.29916666666666|
|2024-10-15 15:07:15|2024-10-12 19:32:51| 67.57333333333334|
|2024-10-20 12:02:18|2024-10-17 17:58:18| 66.06666666666666|
|2024-10-23 12:53:42|2024-10-21 14:28:21|           46.4225|
+-------------------+-------------------+------------------+



In [9]:
spark.sql("""
        SELECT 
                dropoff_datetime,
                pickup_datetime,
                ((CAST(dropoff_datetime AS LONG) - CAST(pickup_datetime AS LONG)) / 3600) AS trip_duration
        FROM
                trip_data

        ORDER BY
                trip_duration DESC
        LIMIT 10          
          """).show()

+-------------------+-------------------+------------------+
|   dropoff_datetime|    pickup_datetime|     trip_duration|
+-------------------+-------------------+------------------+
|2024-10-23 07:40:53|2024-10-16 13:03:49|162.61777777777777|
|2024-10-09 18:06:55|2024-10-03 18:47:25|           143.325|
|2024-10-28 09:46:33|2024-10-22 16:00:55|138.76055555555556|
|2024-10-23 04:43:37|2024-10-18 09:53:32|114.83472222222223|
|2024-10-24 18:30:18|2024-10-21 00:36:24| 89.89833333333333|
|2024-10-24 06:57:38|2024-10-20 13:30:52| 89.44611111111111|
|2024-10-25 14:22:49|2024-10-22 16:04:52| 70.29916666666666|
|2024-10-15 15:07:15|2024-10-12 19:32:51| 67.57333333333334|
|2024-10-20 12:02:18|2024-10-17 17:58:18| 66.06666666666666|
|2024-10-23 12:53:42|2024-10-21 14:28:21|           46.4225|
+-------------------+-------------------+------------------+



**Q5 User Interface**

Spark’s User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040
- 8080



In [10]:
4040

4040

**Q6: Least frequent pickup location zone**

Load the zone lookup data into a temp view in Spark:

wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

- Governor's Island/Ellis Island/Liberty Island
- Arden Heights
- Rikers Island
- Jamaica Bay

In [11]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-05 12:36:29--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.114.39, 52.85.114.54, 52.85.114.114, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.114.39|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.2’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-05 12:36:29 (91.2 MB/s) - ‘taxi_zone_lookup.csv.2’ saved [12331/12331]



In [12]:
schema = types.StructType([
    types.StructField('LocationID',     types.IntegerType(), True),
    types.StructField('Borough',        types.StringType(), True),
    types.StructField('Zone',           types.StringType(), True),
    types.StructField('service_zone',   types.StringType(), True)
])


df_tzl = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('taxi_zone_lookup.csv')



In [13]:
df_tzl.registerTempTable('taxi_zone_lookup')

In [14]:
spark.sql("""

SELECT 
      tzl.Zone,
      COUNT(tzl.Zone) as count
FROM 
      trip_data td
      LEFT JOIN taxi_zone_lookup tzl 
      ON td.PULocationID = tzl.LocationID
GROUP BY
      tzl.Zone
SORT BY
      count ASC

LIMIT 
      10
          """).show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|       Rikers Island|    2|
|       Arden Heights|    2|
|         Jamaica Bay|    3|
| Green-Wood Cemetery|    3|
|Charleston/Totten...|    4|
|   Rossville/Woodrow|    4|
|       West Brighton|    4|
|Eltingville/Annad...|    4|
|       Port Richmond|    4|
+--------------------+-----+

